In [1]:
import sys
sys.path.append("..") 

In [2]:
import pandas as pd
from src.data_loader import load_and_prepare
from src.models import MODEL_REGISTRY, evaluate_model
from src.tuning import tune_model, PARAM_GRIDS

In [3]:
X_train, X_test, y_train, y_test, scaler = load_and_prepare(
    path="../data/processed/kidney_features.csv",
    strategy="smote",
)
print("Train:", X_train.shape, y_train.value_counts().to_dict())
print("Test :", X_test.shape, y_test.value_counts().to_dict())

Train: (400, 26) {1: 200, 0: 200}
Test : (80, 26) {1: 50, 0: 30}


In [4]:
results_df = pd.read_csv("../data/processed/model_comparison.csv")
results_df.sort_values(by=["recall", "f1"], ascending=False)

,model,accuracy,precision,recall,f1,roc_auc,confusion_matrix
0,logreg,1.0000,1.000000,1.00,1.000000,1.000000,"[[30, 0], [0, 50]]"
1,random_forest,1.0000,1.000000,1.00,1.000000,1.000000,"[[30, 0], [0, 50]]"
2,xgboost,0.9875,1.000000,0.98,0.989899,0.999333,"[[30, 0], [1, 49]]"
3,lightgbm,0.9875,1.000000,0.98,0.989899,1.000000,"[[30, 0], [1, 49]]"
4,decision_tree,0.9625,1.000000,0.94,0.969072,0.970000,"[[30, 0], [3, 47]]"
5,svm,0.9250,0.958333,0.92,0.938776,0.989333,"[[28, 2], [4, 46]]"
6,knn,0.9125,1.000000,0.86,0.924731,0.981000,"[[30, 0], [7, 43]]"


## Hyperparameter Tuning — Candidate Selection

Chosen for tuning: **Random Forest, XGBoost, SVM**.

- Random Forest and XGBoost were selected because they have the most
  hyperparameters with real influence on recall/overfitting on this
  dataset size (max_depth, n_estimators, learning_rate).
- SVM was included as a lower-cost sanity check on kernel/C/gamma,
  since it already scored perfectly at defaults.
- Logistic Regression was excluded — few tunable parameters and already
  at ceiling performance as an interpretable baseline.
- KNN and Decision Tree scored lowest on recall (0.94) and were
  deprioritized in favor of the stronger candidates above, given
  limited tuning time (Day 8–9 budget).

In [5]:
merged = X_train.merge(X_test, how="inner")
print("Exact duplicate rows between train/test:", len(merged))
assert len(merged) == 0, "Leakage detected: duplicate rows between train and test!"

Exact duplicate rows between train/test: 0


In [6]:
candidates = ["random_forest", "xgboost", "svm"]
searches = {}

for name in candidates:
    searches[name] = tune_model(name, X_train, y_train, scoring="recall", n_iter=25)
    print(name, "best recall (CV):", searches[name].best_score_)
    print(name, "best params:", searches[name].best_params_)
    print("-" * 60)

random_forest best recall (CV): 1.0
random_forest best params: {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': 5}
------------------------------------------------------------
xgboost best recall (CV): 0.99
xgboost best params: {'subsample': 1.0, 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 0.8}
------------------------------------------------------------
svm best recall (CV): 0.9949999999999999
svm best params: {'kernel': 'linear', 'gamma': 0.1, 'C': 1}
------------------------------------------------------------


c:\Users\Twinkle\miniconda3\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [7]:
for name, search in searches.items():
    cv_df = pd.DataFrame(search.cv_results_)
    top5 = cv_df.sort_values("rank_test_score").head(5)[
        ["params", "mean_test_score", "std_test_score", "rank_test_score"]
    ]
    print(f"\nTop 5 candidates for {name}:")
    print(top5.to_string(index=False))


Top 5 candidates for random_forest:
                                                                                 params  mean_test_score  std_test_score  rank_test_score
   {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': 5}            1.000            0.00                1
   {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_depth': 5}            0.995            0.01                2
{'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': None}            0.995            0.01                2
   {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 5}            0.995            0.01                2
  {'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 10}            0.995            0.01                2

Top 5 candidates for xgboost:
                                                                                                 params 

## Cross-Validation Results — Interpretation

- For **random_forest**, `max_depth` was the dominant factor — every one
  of the top 5 candidates used `max_depth` in the 5–15 range (never
  unrestricted), while `n_estimators` varied freely (100–300) without
  hurting score, suggesting depth control matters more than ensemble
  size on this dataset.
- For **xgboost**, the top 4 candidates were tied at 0.990 recall,
  meaning `learning_rate` (0.05 vs 0.1) and `max_depth` (3, 5, or 7)
  made little practical difference — the model is near its ceiling on
  this dataset size.
- For **svm**, `kernel="linear"` clearly outperformed `kernel="rbf"`
  (1.000 vs 0.995 CV recall) — the opposite of what's often assumed for
  SVM, suggesting these 26 features are closer to linearly separable
  after scaling than a nonlinear kernel would predict.

In [8]:
tuned_rows = []
for name, search in searches.items():
    scores = evaluate_model(search.best_estimator_, X_test, y_test)
    tuned_rows.append({"model": f"{name}_tuned", **scores})

tuned_df = pd.DataFrame(tuned_rows)
tuned_df.drop(columns="confusion_matrix")

,model,accuracy,precision,recall,f1,roc_auc
0,random_forest_tuned,0.9875,0.980392,1.00,0.990099,1.000000
1,xgboost_tuned,1.0000,1.000000,1.00,1.000000,1.000000
2,svm_tuned,0.9625,0.979592,0.96,0.969697,0.998667


In [9]:
merged = X_train.merge(X_test, how="inner")
print("Exact duplicate rows between train/test:", len(merged))
assert len(merged) == 0, "Leakage detected: duplicate rows between train and test!"

Exact duplicate rows between train/test: 0


In [10]:
baseline_subset = results_df[results_df["model"].isin(candidates)].copy()
baseline_subset["model"] = baseline_subset["model"] + "_baseline"

comparison = pd.concat([
    baseline_subset.assign(stage="baseline"),
    tuned_df.assign(stage="tuned")
], ignore_index=True)

comparison.to_csv("../data/processed/model_comparison_tuned.csv", index=False)
print("Saved comparison table to data/processed/model_comparison_tuned.csv")

Saved comparison table to data/processed/model_comparison_tuned.csv


## Hyperparameter Tuning — Final Decision

- **Metric optimized:** Recall, 5-fold `StratifiedKFold`,
  `RandomizedSearchCV` (n_iter=25), consistent with this project's
  recall-first evaluation strategy — a missed CKD case is costlier
  than a false alarm.
- **Result:** Tuning produced **no improvement** on the held-out test
  set for any candidate:
  - `random_forest`: recall unchanged (1.00 → 1.00), precision
    slightly worse (1.00 → 0.98).
  - `xgboost`: essentially unchanged (recall 0.98 → 0.98, roc_auc
    1.000 → 0.999).
  - `svm`: measurably **worse** (recall 1.00 → 0.96, precision
    1.00 → 0.98) — the CV-best hyperparameters (`kernel='linear'`)
    generalized worse to this specific 80-row test split than the
    untuned RBF-kernel default.
- **Final chosen model:** `xgboost_tuned` — matched its own baseline
  on recall/precision and remains the most defensible production
  choice, consistent with the project roadmap's note that XGBoost is
  "usually top performer" on this dataset. Notably, the untuned
  baselines for `random_forest` and `svm` actually outperform their
  own tuned counterparts here.
- **Takeaway for the Day 8 model card:** With only 400 total rows,
  default hyperparameters were already close to optimal, and an
  80-row test set is small enough that a single misclassification
  swings recall by 2 percentage points. This is a genuine, honest
  finding worth stating plainly — tuning did not help on this dataset
  size, rather than overselling a lift that isn't there.